# Fase 6 — Análise SHAP (XGBoost best)  

## Boa prática sênior: análise SHAP como entregável SEPARADO.

Justificativa (padrão FAANG, bancos e fintechs de 1ª linha):

1. **Auditoria e regulador:** SHAP é o artefato de compliance por excelência (LGPD, BCB, B3). 
   Precisa ter notebook próprio, número de versão e checksum de modelo atrelados. Não se mistura com notebook de tuning.
2. **Pesado computacional:** TreeExplainer + Waterfall plots demoram; rodar dentro do notebook de tuning
   faria todo o documento demorar 40min para Run All.
3. **Reprodutibilidade em produção:** O Streamlit carrega os artefatos `.pkl`/`.npy` gerados aqui e explica
   *cada cliente individualmente* em < 10ms, sem re-calcular o SHAP de 37k linhas.

---

Conteúdo deste notebook (execução 1:1 ao script `src/models/fase06_analise_shap.py`):
- [1. Carregar pipeline XGBoost e X_test bruto](#1.-Carregar)
- [2. Transformar X_test bruto → processado (passo 1 e 2 do pipeline)](#2.-Processar)
- [3. TreeExplainer (exato para árvores) → shap_values para todos os 37.500 clientes](#3.-TreeExplainer)
- [4. 📊 Ranking GLOBAL Top 10 por |mean(SHAP)| (RESULTADO PRINCIPAL PEDIDO)](#4.-Ranking-Top10)
- [5. Gráficos (summary beeswarm, bar top10, dependence top4)](#5.-Graficos)
- [6. 3 casos LOCAIS reais do holdout (waterfall: por que cliente foi negado?)](#6.-Casos-Locais)
- [7. Salvar artefatos p/ produção (`.npy` / `.pkl`)](#7.-Salvar-Producao)

Seed = 42. Base de cálculo = SOMENTE HOLDOUT n = 37.500.

In [ ]:
import os, sys, pickle, warnings
import numpy as np, pandas as pd
import shap, matplotlib.pyplot as plt, seaborn as sns
SRC = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), 'src'))
sys.path.insert(0, SRC)
from features.preprocessamento import SEED_DEFAULT
from features.pipeline_modelo import obter_nomes_features
warnings.filterwarnings('ignore')
np.random.seed(SEED_DEFAULT)
shap.initjs()
sns.set_theme(style='whitegrid')
BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
print('SHAP versão:', shap.__version__, ' | SEED =', SEED_DEFAULT)

## 1. Carregar pipeline do XGBoost best e X_test bruto (raw)


In [ ]:
with open(os.path.join(BASE, 'models', '04_xgboost_pipeline.pkl'), 'rb') as f:
    pipe_xgb = pickle.load(f)
X_test_raw = pd.read_csv(os.path.join(BASE, 'data', 'processed', 'X_test.csv'))
y_test     = pd.read_csv(os.path.join(BASE, 'data', 'processed', 'y_test.csv'))['inadimplente_2anos'].astype(int).values
print(f'X_test bruto shape = {X_test_raw.shape}    |    y_test  pos = {int(y_test.sum()):,}  neg = {int((y_test==0).sum()):,}')
X_test_raw.head(3)

## 2. Transformar X_test bruto → processado (passa por "preparo" + "imputacao" do pipeline)

In [ ]:
X_p1 = pipe_xgb.named_steps['preparo'].transform(X_test_raw.copy())
X_p2 = pipe_xgb.named_steps['imputacao'].transform(X_p1)
FEATURE_NAMES = obter_nomes_features(pipe_xgb)
X_test_processado = pd.DataFrame(X_p2, columns = FEATURE_NAMES)
MODELO = pipe_xgb.named_steps['modelo']
print(f'X_test_processado shape = {X_test_processado.shape}')
print(f'Features pós-preparo ({len(FEATURE_NAMES)}):')
for i,n in enumerate(FEATURE_NAMES, 1):
    print(f'  {i:2d}. {n}')
X_test_processado.head(3)

## 3. shap.TreeExplainer — exato (sem aproximações) para árvores


In [ ]:
explicador = shap.TreeExplainer(MODELO)
explanation = explicador(X_test_processado)

if isinstance(explanation, shap.Explanation):
    if len(explanation.values.shape) == 3:
        shap_values_cls1 = explanation.values[:,:,1]
        expected_value_cls1 = float(np.mean(explanation.base_values[:,1])) if len(explanation.base_values.shape)==2 else float(explanation.base_values[1])
    else:
        shap_values_cls1 = explanation.values
        expected_value_cls1 = float(np.mean(explanation.base_values)) if hasattr(explanation.base_values,'__len__') and not np.isscalar(explanation.base_values) else float(explanation.base_values)
elif isinstance(explanation, np.ndarray):
    shap_values_cls1 = explanation[:,:,1] if len(explanation.shape)==3 else explanation
    expected_value_cls1 = float(np.mean(explicador.expected_value)) if hasattr(explicador.expected_value,'__len__') else float(explicador.expected_value)
else:
    raise TypeError('Tipo inesperado: '+str(type(explanation)))

shap_df = pd.DataFrame(shap_values_cls1, columns=FEATURE_NAMES)
print(f'expected_value (log-odds base) = {expected_value_cls1:.5f}')
print(f'shap_values_cls1.shape        = {shap_values_cls1.shape}')

## 4. 📊 RESULTADO PRINCIPAL PEDIDO — Ranking GLOBAL Top 10 por |média(SHAP)|


In [ ]:
mean_abs = shap_df.abs().mean(axis=0).sort_values(ascending=False)
top10 = mean_abs.head(10)
top10_df = pd.DataFrame({'Feature': top10.index, 'Média |SHAP Value|': np.round(top10.values,6)}).set_index(pd.Index(range(1,11),name='Rank'))
top10_df

In [ ]:
detalhe = []
for i, feat in enumerate(top10.index, 1):
    detalhe.append(dict(
        Rank=i, Feature=feat,
        MédAbsSHAP=round(top10.loc[feat],6),
        MédiaSHAP_sinal=round(float(shap_df[feat].mean()), 6),
        Máx_↑risco=round(float(shap_df[feat].max()),4),
        Máx_↓risco=round(float(shap_df[feat].min()),4),
        PCT_↑_risco=f"{100.*(shap_df[feat]>0).mean():.2f}%"
    ))
pd.DataFrame(detalhe).set_index('Rank')

## 5. Gráficos SHAP

Gráficos gravados em `reports/figures/` (todos os resultados do notebook serão gravados também ao final).

- 📊 **Beeswarm summary top10** (cor = valor da feature; eixo x = impacto no log-odds de calote).
- 📊 **Bar top10** (ranking de importância global).
- 📊 **Dependence Plot** das top4 features.


In [ ]:
FIG_DIR = os.path.join(BASE, 'reports', 'figures')
# 5a. Beeswarm top10
fig = plt.figure(figsize=(12,9))
shap.summary_plot(shap_df[top10.index].values,
                  X_test_processado[top10.index].values,
                  feature_names=top10.index.tolist(),
                  max_display=10, show=False)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fase06_shap_summary_beeswarm_top10_holdout.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
# 5b. Bar top10
fig = plt.figure(figsize=(10,7))
shap.summary_plot(shap_df[top10.index].values,
                  X_test_processado[top10.index].values,
                  feature_names=top10.index.tolist(),
                  plot_type='bar', max_display=10, show=False)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fase06_shap_bar_top10_holdout.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
# 5c. Dependence plots das top 4
for feat in top10.index[:4]:
    fig = plt.figure(figsize=(10,6))
    shap.dependence_plot(feat, shap_df.values, X_test_processado, feature_names=FEATURE_NAMES, show=False)
    plt.title(f'Dependence Plot: {feat}')
    plt.tight_layout()
    safe = feat.replace('/','_').replace('(','_').replace(')','_').replace(' ','_')
    fig.savefig(os.path.join(FIG_DIR, f'fase06_shap_dependence_{safe}_holdout.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
'✓ Gráficos beeswarm + bar + 4× dependence salvos em reports/figures/'

## 6. 3 casos LOCAIS reais do holdout (waterfall plot — "por que o cliente foi negado?")

  (A) Calote real — modelo NEGOU (acerto, score alto)

  (B) Adimplente real — modelo APROVOU (acerto, score baixo)

  (C) Borderline — score próximo do threshold ótimo 0,56 (requer análise manual)

In [ ]:
probas = pipe_xgb.predict_proba(X_test_raw)[:, 1]
TH = 0.56
preds  = (probas >= TH).astype(int)
aux = pd.DataFrame({'y_real':y_test,'proba':probas,'pred':preds,
                    'TP':(preds==1)&(y_test==1), 'TN':(preds==0)&(y_test==0)})
tp_idx = aux[aux['TP'] & (probas > 0.86)].head(1).index[0]
tn_idx = aux[aux['TN'] & (probas < 0.12)].head(1).index[0]
bd_idx = aux[(probas >= TH-0.02) & (probas <= TH+0.02)].head(1).index[0]
CASOS = [('A_CALOTE_NEGADO_TP', tp_idx), ('B_ADIMPLENTE_APROVADO_TN', tn_idx), ('C_BORDERLINE', bd_idx)]
CASOS

In [ ]:
for rotulo, idx in CASOS:
    print(f'\n► {rotulo}: idx={idx}, proba={probas[idx]:.3f}, pred={preds[idx]}, y_real={y_test[idx]}')
    exp = shap.Explanation(values=shap_df.iloc[idx].values, base_values=expected_value_cls1,
                            data=X_test_processado.iloc[idx].values, feature_names=FEATURE_NAMES)
    fig = plt.figure(figsize=(11,7))
    shap.waterfall_plot(exp, max_display=10, show=False)
    plt.title(f'Cliente {rotulo} — proba={probas[idx]:.3f}')
    plt.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f'fase06_shap_waterfall_{rotulo}_holdout.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
    d = pd.DataFrame({'Feature':FEATURE_NAMES,'Valor':np.round(X_test_processado.iloc[idx].values,4),'SHAP':np.round(exp.values,5)})
    d['|SHAP|'] = d['SHAP'].abs(); d = d.sort_values('|SHAP|').tail(10).drop('|SHAP|',axis=1,).sort_values('SHAP',ascending=False)
    print(d.to_string(index=False))

## 7. Salvar artefatos para produção (Streamlit)

  · `models/shap_values_holdout_XGBbest.npy` — matriz bruta para auditoria offline.

  · `models/shap_artefatos_producao.pkl` — feature_names, expected_value, threshold e nota de como explicar 1 cliente novo sem recarregar tudo.


In [ ]:
MODEL_DIR = os.path.join(BASE, 'models')
np.save(os.path.join(MODEL_DIR, 'shap_values_holdout_XGBbest.npy'), shap_values_cls1)
with open(os.path.join(MODEL_DIR, 'shap_artefatos_producao.pkl'),'wb') as f:
    pickle.dump({
        'feature_names': FEATURE_NAMES,
        'expected_value_logodds': float(expected_value_cls1),
        'top10_features': top10_df['Feature'].tolist(),
        'top10_mean_abs_shap': top10_df['Média |SHAP Value|'].tolist(),
        'TH_OTIMO_XGB_5000_500': TH,
        'CUSTO_FN_RS': 5000, 'CUSTO_FP_RS': 500,
        'modelo_tipo': 'XGBClassifier (TreeExplainer exato)',
    }, f)
'✓ Artefatos SHAP salvos em models/'